# 🔬 Multi-Agent AI Deep Researcher
### Group 15 — AI Agent Project

**Pipeline:** `Query → Query Planning → Retrieval → Analysis → Insight Generation → Report Builder`

**Agents:**
| Agent | Role | Technology |
|---|---|---|
| Query Planner | Decomposes topic into 4–6 targeted sub-queries | LangChain + OpenRouter |
| Contextual Retriever | Fetches web sources + PDF semantic search | Tavily API + ChromaDB |
| Critical Analysis | Summarises, scores credibility, detects contradictions | GPT via OpenRouter |
| Insight Generator | Synthesises hypotheses & trends | GPT via OpenRouter |
| Report Builder | Compiles structured Markdown report | GPT via OpenRouter |
| Orchestrator | Manages state flow between agents | LangGraph |

**UI:** Gradio

## 📦 Step 1: Install Dependencies

In [ ]:
%%capture
!pip install langchain langchain-openai langchain-community langgraph
!pip install tavily-python chromadb pypdf langchain-text-splitters
!pip install gradio openai tiktoken
print("✅ All dependencies installed!")

In [ ]:
%%capture
!pip install SpeechRecognition pydub
print("✅ Speech-to-text dependencies installed!")

## 🔑 Step 2: API Keys Setup

You need:
- **OpenRouter API Key** → https://openrouter.ai (free tier available)
- **Tavily API Key** → https://tavily.com (free tier: 1000 calls/month)

In [ ]:
import os
from getpass import getpass

print("🔑 Enter your API Keys (input is hidden)")
print("-" * 45)

OPENROUTER_API_KEY = getpass("OpenRouter API Key: ")
TAVILY_API_KEY     = getpass("Tavily API Key:     ")

os.environ["OPENAI_API_KEY"]      = OPENROUTER_API_KEY
os.environ["OPENAI_API_BASE"]     = "https://openrouter.ai/api/v1"
os.environ["TAVILY_API_KEY"]      = TAVILY_API_KEY

print("\n✅ API keys set successfully!")

🔑 Enter your API Keys (input is hidden)
---------------------------------------------
OpenRouter API Key: ··········
Tavily API Key:     ··········

✅ API keys set successfully!


## 🧠 Step 3: Core Imports & LLM Setup

In [ ]:
import json
import time
import textwrap
from typing import TypedDict, List, Optional, Annotated
from datetime import datetime

# LangChain
from langchain_openai import ChatOpenAI, OpenAIEmbeddings
from langchain_core.messages import HumanMessage, SystemMessage # Changed from langchain.schema
from langchain_community.tools.tavily_search import TavilySearchResults
from langchain_community.document_loaders import PyPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_community.vectorstores import Chroma

# LangGraph
from langgraph.graph import StateGraph, END
import operator

# ── LLM via OpenRouter ──────────────────────────────────────────────────────
llm = ChatOpenAI(
    model="openai/gpt-4o-mini",           # change to gpt-4o for higher quality
    openai_api_key=OPENROUTER_API_KEY,
    openai_api_base="https://openrouter.ai/api/v1",
    temperature=0.3,
    max_tokens=2048,
)

# ── Embeddings via OpenRouter ────────────────────────────────────────────────
embeddings = OpenAIEmbeddings(
    model="openai/text-embedding-ada-002",
    openai_api_key=OPENROUTER_API_KEY,
    openai_api_base="https://openrouter.ai/api/v1",
)

# ── Tavily Web Search Tool ───────────────────────────────────────────────────
tavily_tool = TavilySearchResults(max_results=5)

print("✅ LLM, Embeddings, and Tavily search tool ready!")

/usr/local/lib/python3.12/dist-packages/langgraph/cache/base/__init__.py:8: LangChainPendingDeprecationWarning: The default value of `allowed_objects` will change in a future version. Pass an explicit value (e.g., allowed_objects='messages' or allowed_objects='core') to suppress this warning.
  from langgraph.checkpoint.serde.jsonplus import JsonPlusSerializer


✅ LLM, Embeddings, and Tavily search tool ready!


/tmp/ipykernel_15022/3015185497.py:36: LangChainDeprecationWarning: The class `TavilySearchResults` was deprecated in LangChain 0.3.25 and will be removed in 1.0. An updated version of the class exists in the `langchain-tavily package and should be used instead. To use it run `pip install -U `langchain-tavily` and import as `from `langchain_tavily import TavilySearch``.
  tavily_tool = TavilySearchResults(max_results=5)


## 🗂️ Step 4: LangGraph State Schema

In [ ]:
class ResearchState(TypedDict):
    """Shared state that flows through all agents in the LangGraph pipeline."""
    # Input
    topic:              str
    pdf_paths:          List[str]          # paths to any uploaded PDFs

    # Agent outputs
    sub_queries:        List[str]          # Query Planner
    raw_sources:        List[dict]         # Retriever (web)
    pdf_excerpts:       List[str]          # Retriever (PDF)
    validated_sources:  List[dict]         # Source Validation
    contradictions:     List[str]          # Contradiction Detection
    summary:            str                # Critical Analysis
    insights:           str                # Insight Generator
    final_report:       str                # Report Builder

    # Progress log (printed to console → shows agentic activity)
    log:                Annotated[List[str], operator.add]

print("✅ State schema defined!")

✅ State schema defined!


## 🤖 Step 5: Agent Definitions

Each agent is a pure function `(state) → partial state update`.  
All console prints show **agentic activity** in real time.

In [ ]:
# ══════════════════════════════════════════════════════════════════════════════
# AGENT 1 — Query Planner
# Decomposes the research topic into 4-6 targeted sub-queries (multi-hop)
# ══════════════════════════════════════════════════════════════════════════════
def query_planner_agent(state: ResearchState) -> dict:
    print("\n" + "="*60)
    print("🧭 [AGENT 1] QUERY PLANNER — Decomposing research topic...")
    print("="*60)

    topic = state["topic"]
    print(f"   Topic: {topic}")

    messages = [
        SystemMessage(content=(
            "You are a research query strategist. "
            "Given a research topic, decompose it into exactly 5 targeted sub-queries "
            "that together give multi-hop, comprehensive coverage. "
            "Return ONLY a JSON array of 5 strings, no extra text."
        )),
        HumanMessage(content=f"Research topic: {topic}")
    ]

    response = llm.invoke(messages)
    raw = response.content.strip()

    # Parse JSON safely
    try:
        if "```" in raw:
            raw = raw.split("```")[1].replace("json", "").strip()
        sub_queries = json.loads(raw)
    except Exception:
        # Fallback: split by newline
        sub_queries = [line.strip().lstrip("0123456789.-) ") for line in raw.split("\n") if line.strip()][:5]

    print(f"   ✅ Generated {len(sub_queries)} sub-queries:")
    for i, q in enumerate(sub_queries, 1):
        print(f"      {i}. {q}")

    return {
        "sub_queries": sub_queries,
        "log": [f"[Query Planner] Generated {len(sub_queries)} sub-queries for: {topic}"]
    }


# ══════════════════════════════════════════════════════════════════════════════
# AGENT 2 — Contextual Retriever (Web + PDF)
# Fetches web sources via Tavily and extracts PDF content via ChromaDB
# ══════════════════════════════════════════════════════════════════════════════
def retriever_agent(state: ResearchState) -> dict:
    print("\n" + "="*60)
    print("📡 [AGENT 2] CONTEXTUAL RETRIEVER — Fetching sources...")
    print("="*60)

    sub_queries  = state["sub_queries"]
    pdf_paths    = state.get("pdf_paths", [])
    all_sources  = []
    pdf_excerpts = []

    # ── Web Search via Tavily ────────────────────────────────────────────────
    print("   🌐 Web search via Tavily...")
    for i, query in enumerate(sub_queries, 1):
        print(f"      Searching ({i}/{len(sub_queries)}): {query[:70]}...")
        try:
            results = tavily_tool.invoke(query)
            for r in results:
                all_sources.append({
                    "query":   query,
                    "title":   r.get("title", "Unknown"),
                    "url":     r.get("url", ""),
                    "content": r.get("content", "")[:1500],  # cap at 1500 chars per source
                    "score":   r.get("score", 0),
                })
            time.sleep(0.5)   # rate-limit courtesy
        except Exception as e:
            print(f"      ⚠️  Tavily error for query {i}: {e}")

    print(f"   ✅ Retrieved {len(all_sources)} web sources.")

    # ── PDF Semantic Search via ChromaDB ────────────────────────────────────
    if pdf_paths:
        print(f"   📄 Processing {len(pdf_paths)} PDF(s) with ChromaDB...")
        splitter = RecursiveCharacterTextSplitter(chunk_size=800, chunk_overlap=100)
        docs = []
        for path in pdf_paths:
            try:
                loader = PyPDFLoader(path)
                pages  = loader.load()
                chunks = splitter.split_documents(pages)
                docs.extend(chunks)
                print(f"      Loaded: {path} ({len(chunks)} chunks)")
            except Exception as e:
                print(f"      ⚠️  PDF load error: {e}")

        if docs:
            vectorstore = Chroma.from_documents(docs, embeddings)
            for query in sub_queries[:3]:   # query top-3 sub-queries against PDFs
                results = vectorstore.similarity_search(query, k=3)
                for r in results:
                    pdf_excerpts.append(f"[PDF] {r.page_content[:600]}")
            print(f"   ✅ Extracted {len(pdf_excerpts)} PDF excerpts.")
    else:
        print("   ℹ️  No PDFs uploaded — skipping PDF retrieval.")

    return {
        "raw_sources":  all_sources,
        "pdf_excerpts": pdf_excerpts,
        "log": [f"[Retriever] {len(all_sources)} web sources + {len(pdf_excerpts)} PDF excerpts retrieved."]
    }


# ══════════════════════════════════════════════════════════════════════════════
# AGENT 3 — Source Validator
# Scores each source for relevance & credibility; filters out weak ones
# ══════════════════════════════════════════════════════════════════════════════
def source_validator_agent(state: ResearchState) -> dict:
    print("\n" + "="*60)
    print("🔍 [AGENT 3] SOURCE VALIDATOR — Scoring & filtering sources...")
    print("="*60)

    sources = state["raw_sources"]
    topic   = state["topic"]

    if not sources:
        print("   ⚠️  No sources to validate.")
        return {"validated_sources": [], "log": ["[Validator] No sources to validate."]}

    # Send all source titles+snippets in one LLM call for efficiency
    source_list = "\n".join(
        [f"{i+1}. [{s['title']}] {s['content'][:300]}" for i, s in enumerate(sources[:20])]
    )

    messages = [
        SystemMessage(content=(
            "You are a research librarian. For each source, output a JSON array where each element has: "
            '"index" (1-based), "relevance" (0-10), "credibility" (0-10), "keep" (true/false). '
            "Keep a source if relevance >= 5 AND credibility >= 4. Return ONLY the JSON array."
        )),
        HumanMessage(content=f"Research topic: {topic}\n\nSources:\n{source_list}")
    ]

    response  = llm.invoke(messages)
    raw       = response.content.strip()

    try:
        if "```" in raw:
            raw = raw.split("```")[1].replace("json", "").strip()
        scores = json.loads(raw)
    except Exception:
        scores = [{"index": i+1, "keep": True} for i in range(len(sources))]

    keep_indices    = {s["index"]-1 for s in scores if s.get("keep", True)}
    validated       = [s for i, s in enumerate(sources) if i in keep_indices]

    print(f"   ✅ Kept {len(validated)}/{len(sources)} sources after validation.")
    for s in validated[:5]:
        print(f"      • {s['title'][:70]}")

    return {
        "validated_sources": validated,
        "log": [f"[Validator] Kept {len(validated)}/{len(sources)} sources."]
    }


# ══════════════════════════════════════════════════════════════════════════════
# AGENT 4 — Critical Analysis (Summariser + Contradiction Detector)
# Reflection design pattern: summarise → then look for contradictions
# ══════════════════════════════════════════════════════════════════════════════
def critical_analysis_agent(state: ResearchState) -> dict:
    print("\n" + "="*60)
    print("🧐 [AGENT 4] CRITICAL ANALYSIS — Summarising & finding contradictions...")
    print("="*60)

    sources      = state["validated_sources"]
    pdf_excerpts = state.get("pdf_excerpts", [])
    topic        = state["topic"]

    # Build a condensed context (cap at ~6000 chars to stay within context window)
    context_parts = []
    for s in sources:
        context_parts.append(f"Source: {s['title']}\n{s['content'][:400]}")
    context_parts.extend(pdf_excerpts[:5])
    context = "\n\n".join(context_parts)[:6000]

    # ── Summarisation (Reflection) ───────────────────────────────────────────
    print("   📝 Generating summary...")
    sum_messages = [
        SystemMessage(content=(
            "You are a critical research analyst. "
            "Summarise the key findings from the provided sources in 4-6 concise paragraphs. "
            "Highlight the most important facts, data points, and expert opinions. "
            "Be objective and accurate."
        )),
        HumanMessage(content=f"Topic: {topic}\n\nSources:\n{context}")
    ]
    summary = llm.invoke(sum_messages).content
    print("   ✅ Summary generated.")

    # ── Contradiction Detection ──────────────────────────────────────────────
    print("   ⚡ Detecting contradictions...")
    con_messages = [
        SystemMessage(content=(
            "You are a fact-checker. Review the research sources and identify "
            "any contradictions, conflicting claims, or disputed facts. "
            "List each contradiction as a bullet point. If none found, say 'No major contradictions found.'"
        )),
        HumanMessage(content=f"Topic: {topic}\n\nSources:\n{context}")
    ]
    contradictions_text = llm.invoke(con_messages).content
    contradictions      = [line.strip() for line in contradictions_text.split("\n") if line.strip()]

    print(f"   ✅ Found {len(contradictions)} contradiction note(s).")

    return {
        "summary":       summary,
        "contradictions": contradictions,
        "log": [f"[Critical Analysis] Summary done. {len(contradictions)} contradictions noted."]
    }


# ══════════════════════════════════════════════════════════════════════════════
# AGENT 5 — Insight Generator
# Synthesises hypotheses, trends, and key takeaways using reasoning chains
# ══════════════════════════════════════════════════════════════════════════════
def insight_generator_agent(state: ResearchState) -> dict:
    print("\n" + "="*60)
    print("💡 [AGENT 5] INSIGHT GENERATOR — Synthesising trends & hypotheses...")
    print("="*60)

    summary        = state["summary"]
    contradictions = state["contradictions"]
    topic          = state["topic"]

    messages = [
        SystemMessage(content=(
            "You are a strategic research analyst. Based on the research summary and contradictions, "
            "generate: (1) 3 key insights/trends, (2) 2 testable hypotheses, "
            "(3) 3 actionable recommendations, and (4) future research directions. "
            "Use clear reasoning chains. Format with Markdown headers."
        )),
        HumanMessage(content=(
            f"Topic: {topic}\n\n"
            f"Summary:\n{summary}\n\n"
            f"Contradictions:\n{'\n'.join(contradictions[:10])}"
        ))
    ]

    insights = llm.invoke(messages).content
    print("   ✅ Insights & hypotheses generated.")

    return {
        "insights": insights,
        "log": ["[Insight Generator] Trends, hypotheses, and recommendations synthesised."]
    }


# ══════════════════════════════════════════════════════════════════════════════
# AGENT 6 — Report Builder
# Compiles all agent outputs into a structured Markdown research report
# ══════════════════════════════════════════════════════════════════════════════
def report_builder_agent(state: ResearchState) -> dict:
    print("\n" + "="*60)
    print("📄 [AGENT 6] REPORT BUILDER — Compiling final report...")
    print("="*60)

    topic          = state["topic"]
    sub_queries    = state["sub_queries"]
    validated      = state["validated_sources"]
    summary        = state["summary"]
    contradictions = state["contradictions"]
    insights       = state["insights"]
    log            = state.get("log", [])
    timestamp      = datetime.now().strftime("%Y-%m-%d %H:%M")

    # Build reference list
    references = ""
    for i, s in enumerate(validated[:10], 1):
        references += f"{i}. [{s['title']}]({s['url']})\n"

    messages = [
        SystemMessage(content=(
            "You are a professional research report writer. "
            "Compile the provided materials into a polished, structured Markdown report. "
            "Use the exact section headers provided. Be thorough but concise."
        )),
        HumanMessage(content=(
            f"Create a report with these sections:\n"
            f"1. Executive Summary\n"
            f"2. Research Scope & Methodology\n"
            f"3. Key Findings\n"
            f"4. Contradictions & Debates\n"
            f"5. Strategic Insights & Recommendations\n"
            f"6. Conclusion\n\n"
            f"Topic: {topic}\n\n"
            f"Sub-queries used: {json.dumps(sub_queries, indent=2)}\n\n"
            f"Summary:\n{summary}\n\n"
            f"Contradictions:\n{'\n'.join(contradictions[:10])}\n\n"
            f"Insights:\n{insights}"
        ))
    ]

    report_body = llm.invoke(messages).content

    final_report = f"""# 🔬 Deep Research Report
**Topic:** {topic}
**Generated:** {timestamp}
**Sources analysed:** {len(validated)}

---

{report_body}

---

## 📚 References
{references}

---

## 🤖 Agent Activity Log
{'\n'.join(['- ' + l for l in log])}
"""

    print("   ✅ Final report compiled successfully!")
    print("\n" + "="*60)
    print("🎉 PIPELINE COMPLETE")
    print("="*60)

    return {
        "final_report": final_report,
        "log": [f"[Report Builder] Final report compiled. {len(validated)} sources cited."]
    }

print("✅ All 6 agents defined!")

✅ All 6 agents defined!


## 🕸️ Step 6: Build the LangGraph Orchestration Pipeline

In [ ]:
# ── Build LangGraph ──────────────────────────────────────────────────────────
workflow = StateGraph(ResearchState)

# Register nodes (agents)
workflow.add_node("query_planner",    query_planner_agent)
workflow.add_node("retriever",        retriever_agent)
workflow.add_node("source_validator", source_validator_agent)
workflow.add_node("critical_analysis",critical_analysis_agent)
workflow.add_node("insight_generator",insight_generator_agent)
workflow.add_node("report_builder",   report_builder_agent)

# Define the sequential pipeline
# Query → Retriever → Validator → Analysis → Insight → Report
workflow.set_entry_point("query_planner")
workflow.add_edge("query_planner",    "retriever")
workflow.add_edge("retriever",        "source_validator")
workflow.add_edge("source_validator", "critical_analysis")
workflow.add_edge("critical_analysis","insight_generator")
workflow.add_edge("insight_generator","report_builder")
workflow.add_edge("report_builder",   END)

# Compile
app = workflow.compile()

print("✅ LangGraph pipeline compiled!")
print()
print("Pipeline flow:")
print("  Query Planner → Retriever → Source Validator")
print("  → Critical Analysis → Insight Generator → Report Builder")

✅ LangGraph pipeline compiled!

Pipeline flow:
  Query Planner → Retriever → Source Validator
  → Critical Analysis → Insight Generator → Report Builder


## 🖥️ Step 7: Gradio UI

Run this cell to launch the interactive web interface. You can:
- Enter any research topic
- Optionally upload PDF files for semantic search
- Watch agent activity in real time
- Download the final Markdown report

In [ ]:
import gradio as gr
import tempfile
import os

def run_research(topic: str, pdf_files):
    """Main function called by Gradio UI."""
    if not topic.strip():
        return "❌ Please enter a research topic.", ""

    # Handle uploaded PDFs
    pdf_paths = []
    if pdf_files:
        for f in pdf_files:
            pdf_paths.append(f.name)

    # Build initial state
    initial_state: ResearchState = {
        "topic":             topic.strip(),
        "pdf_paths":         pdf_paths,
        "sub_queries":       [],
        "raw_sources":       [],
        "pdf_excerpts":      [],
        "validated_sources": [],
        "contradictions":    [],
        "summary":           "",
        "insights":          "",
        "final_report":      "",
        "log":               [],
    }

    print(f"\n🚀 Starting research pipeline for: '{topic}'")
    print(f"   PDFs: {len(pdf_paths)} file(s) provided")

    try:
        result = app.invoke(initial_state)
        report = result.get("final_report", "No report generated.")

        # Save report to a temp .md file for download
        tmp = tempfile.NamedTemporaryFile(delete=False, suffix=".md", mode="w", encoding="utf-8")
        tmp.write(report)
        tmp.close()

        return report, tmp.name

    except Exception as e:
        error_msg = f"❌ Pipeline error: {str(e)}\n\nPlease check your API keys and try again."
        return error_msg, ""


# ── Gradio Interface ─────────────────────────────────────────────────────────
with gr.Blocks(
    title="Multi-Agent AI Deep Researcher",
    theme=gr.themes.Soft(),
    css=".gradio-container { max-width: 1100px !important; }"
) as demo:

    gr.Markdown("""
    # 🔬 Multi-Agent AI Deep Researcher
    ### Powered by LangGraph · Tavily · OpenRouter · ChromaDB

    **Pipeline:** Query Planner → Retriever → Source Validator → Critical Analysis → Insight Generator → Report Builder
    """)

    with gr.Row():
        with gr.Column(scale=2):
            topic_input = gr.Textbox(
                label="🔎 Research Topic",
                placeholder="e.g. Impact of AI on healthcare diagnostics in 2024",
                lines=2,
            )
            pdf_input = gr.File(
                label="📁 Upload PDFs (optional — for semantic search)",
                file_types=[".pdf"],
                file_count="multiple",
            )
            run_btn = gr.Button("🚀 Run Deep Research", variant="primary", size="lg")

        with gr.Column(scale=1):
            gr.Markdown("""
            ### 🤖 Agent Pipeline
            1. **Query Planner** — Decomposes topic into 5 sub-queries
            2. **Retriever** — Fetches web + PDF sources
            3. **Source Validator** — Scores & filters sources
            4. **Critical Analysis** — Summarises & finds contradictions
            5. **Insight Generator** — Synthesises trends & hypotheses
            6. **Report Builder** — Compiles final report

            > ℹ️ **Note:** Research takes ~2–4 minutes.
            > Watch the Colab console for live agent activity!
            """)

    gr.Markdown("---")

    report_output = gr.Markdown(
        label="📄 Research Report",
        value="*Your report will appear here after running the pipeline...*"
    )

    download_btn = gr.File(label="⬇️ Download Report (.md)")

    # Example topics
    gr.Examples(
        examples=[
            ["Impact of large language models on software engineering productivity"],
            ["Climate change adaptation strategies for coastal cities in 2025"],
            ["Quantum computing applications in drug discovery"],
            ["The future of autonomous vehicles and urban mobility"],
        ],
        inputs=[topic_input],
    )

    # Wire up
    run_btn.click(
        fn=run_research,
        inputs=[topic_input, pdf_input],
        outputs=[report_output, download_btn],
        show_progress=True,
    )

# Launch
demo.launch(share=True, debug=False)

/tmp/ipykernel_15022/1435950568.py:51: DeprecationWarning: The 'theme' parameter in the Blocks constructor will be removed in Gradio 6.0. You will need to pass 'theme' to Blocks.launch() instead.
  with gr.Blocks(
/tmp/ipykernel_15022/1435950568.py:51: DeprecationWarning: The 'css' parameter in the Blocks constructor will be removed in Gradio 6.0. You will need to pass 'css' to Blocks.launch() instead.
  with gr.Blocks(


Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://1480f641f90039afb9.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


In [ ]:
import speech_recognition as sr
from pydub import AudioSegment
import tempfile
import os
import gradio as gr

def transcribe_audio(audio_filepath):
    if audio_filepath is None:
        return ""

    recognizer = sr.Recognizer()
    try:
        # Convert audio to a format SpeechRecognition can handle (e.g., WAV)
        audio = AudioSegment.from_file(audio_filepath)
        # Create a temporary WAV file
        with tempfile.NamedTemporaryFile(suffix=".wav", delete=False) as temp_wav:
            audio.export(temp_wav.name, format="wav")
            temp_wav_path = temp_wav.name

        with sr.AudioFile(temp_wav_path) as source:
            audio_data = recognizer.record(source)
            text = recognizer.recognize_google(audio_data)
            os.remove(temp_wav_path) # Clean up temp file
            return text
    except sr.UnknownValueError:
        return "Could not understand audio"
    except sr.RequestError as e:
        return f"Speech recognition service error; {e}"
    except Exception as e:
        return f"Error processing audio: {e}"

def run_research(topic: str, pdf_files):
    """Main function called by Gradio UI."""
    if not topic.strip():
        return "❌ Please enter a research topic.", ""

    # Handle uploaded PDFs
    pdf_paths = []
    if pdf_files:
        for f in pdf_files:
            pdf_paths.append(f.name)

    # Build initial state
    initial_state: ResearchState = {
        "topic":             topic.strip(),
        "pdf_paths":         pdf_paths,
        "sub_queries":       [],
        "raw_sources":       [],
        "pdf_excerpts":      [],
        "validated_sources": [],
        "contradictions":    [],
        "summary":           "",
        "insights":          "",
        "final_report":      "",
        "log":               [],
    }

    print(f"\n🚀 Starting research pipeline for: '{topic}'")
    print(f"   PDFs: {len(pdf_paths)} file(s) provided")

    try:
        result = app.invoke(initial_state)
        report = result.get("final_report", "No report generated.")

        # Save report to a temp .md file for download
        tmp = tempfile.NamedTemporaryFile(delete=False, suffix=".md", mode="w", encoding="utf-8")
        tmp.write(report)
        tmp.close()

        return report, tmp.name

    except Exception as e:
        error_msg = f"❌ Pipeline error: {str(e)}\n\nPlease check your API keys and try again."
        return error_msg, ""


# ── Gradio Interface ─────────────────────────────────────────────────────────
with gr.Blocks(
    title="Multi-Agent AI Deep Researcher",
    theme=gr.themes.Soft(),
    css=".gradio-container { max-width: 1100px !important; }"
) as demo:

    gr.Markdown("""
    # 🔬 Multi-Agent AI Deep Researcher
    ### Powered by LangGraph · Tavily · OpenRouter · ChromaDB

    **Pipeline:** Query Planner → Retriever → Source Validator → Critical Analysis → Insight Generator → Report Builder
    """)

    with gr.Row():
        with gr.Column(scale=2):
            topic_input = gr.Textbox(
                label="🔎 Research Topic",
                placeholder="e.g. Impact of AI on healthcare diagnostics in 2024",
                lines=2,
            )
            # Add microphone input and transcribe button
            audio_input = gr.Audio(
                sources=["microphone"],
                type="filepath",
                label="🎤 Speak your research topic"
            )
            transcribe_btn = gr.Button("🗣️ Transcribe Voice")

            pdf_input = gr.File(
                label="📁 Upload PDFs (optional — for semantic search)",
                file_types=[".pdf"],
                file_count="multiple",
            )
            run_btn = gr.Button("🚀 Run Deep Research", variant="primary", size="lg")

        with gr.Column(scale=1):
            gr.Markdown("""
            ### 🤖 Agent Pipeline
            1. **Query Planner** — Decomposes topic into 5 sub-queries
            2. **Retriever** — Fetches web + PDF sources
            3. **Source Validator** — Scores & filters sources
            4. **Critical Analysis** — Summarises & finds contradictions
            5. **Insight Generator** — Synthesises trends & hypotheses
            6. **Report Builder** — Compiles final report

            > ℹ️ **Note:** Research takes ~2–4 minutes.
            > Watch the Colab console for live agent activity!
            """)

    gr.Markdown("---")

    report_output = gr.Markdown(
        label="📄 Research Report",
        value="*Your report will appear here after running the pipeline..."
    )

    download_btn = gr.File(label="⬇️ Download Report (.md)")

    # Example topics
    gr.Examples(
        examples=[
            ["Impact of large language models on software engineering productivity"],
            ["Climate change adaptation strategies for coastal cities in 2025"],
            ["Quantum computing applications in drug discovery"],
            ["The future of autonomous vehicles and urban mobility"],
        ],
        inputs=[topic_input],
    )

    # Wire up transcription
    transcribe_btn.click(
        fn=transcribe_audio,
        inputs=[audio_input],
        outputs=[topic_input],
        show_progress=True,
    )

    # Wire up research run
    run_btn.click(
        fn=run_research,
        inputs=[topic_input, pdf_input],
        outputs=[report_output, download_btn],
        show_progress=True,
    )

# Launch
demo.launch(share=True, debug=False)

/tmp/ipykernel_15022/332624790.py:78: DeprecationWarning: The 'theme' parameter in the Blocks constructor will be removed in Gradio 6.0. You will need to pass 'theme' to Blocks.launch() instead.
  with gr.Blocks(
/tmp/ipykernel_15022/332624790.py:78: DeprecationWarning: The 'css' parameter in the Blocks constructor will be removed in Gradio 6.0. You will need to pass 'css' to Blocks.launch() instead.
  with gr.Blocks(


Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://2df1082b533b8d2e42.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


---
## 🧪 (Optional) Step 8: Run Pipeline Directly Without UI

Use this cell to test without Gradio — useful for debugging or scripted runs.

In [ ]:
%%capture
!pip install langchain==0.1.16 langchain-openai==0.1.3 langchain-core==0.1.47 pydantic==1.10.13 \
             langchain-community==0.0.32 langgraph==0.0.51 tavily-python==1.1.0 chromadb==0.4.24 \
             pypdf==4.2.0 langchain-text-splitters==0.0.1

# Add necessary imports for ResearchState definition
from typing import TypedDict, List, Annotated
import operator

# Add necessary imports for SystemMessage and HumanMessage
from langchain_core.messages import HumanMessage, SystemMessage
from datetime import datetime # datetime is used by report_builder_agent
import json # json is used by query_planner_agent
import time # time is used by retriever_agent

# LangChain
from langchain_openai import ChatOpenAI, OpenAIEmbeddings
from langchain_community.tools.tavily_search import TavilySearchResults
from langchain_community.document_loaders import PyPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_community.vectorstores import Chroma

# LangGraph
from langgraph.graph import StateGraph, END

# Import os to get API keys from environment variables
import os

# ── LLM via OpenRouter (copied for self-containment) ──────────────────────────────────────────────────────
llm = ChatOpenAI(
    model="openai/gpt-4o-mini",           # change to gpt-4o for higher quality
    openai_api_key=os.environ.get("OPENAI_API_KEY"), # Use API key from environment
    openai_api_base="https://openrouter.ai/api/v1",
    temperature=0.3,
    max_tokens=2048,
)

# ── Embeddings via OpenRouter (copied for self-containment) ────────────────────────────────────────────────
embeddings = OpenAIEmbeddings(
    model="openai/text-embedding-ada-002",
    openai_api_key=os.environ.get("OPENAI_API_KEY"), # Use API key from environment
    openai_api_base="https://openrouter.ai/api/v1",
)

# ── Tavily Web Search Tool (copied for self-containment) ───────────────────────────────────────────────────
tavily_tool = TavilySearchResults(max_results=5)

# Definition of ResearchState (copied for self-containment)
class ResearchState(TypedDict):
    """Shared state that flows through all agents in the LangGraph pipeline."""
    # Input
    topic:              str
    pdf_paths:          List[str]          # paths to any uploaded PDFs

    # Agent outputs
    sub_queries:        List[str]          # Query Planner
    raw_sources:        List[dict]         # Retriever (web)
    pdf_excerpts:       List[str]          # Retriever (PDF)
    validated_sources:  List[dict]         # Source Validation
    contradictions:     List[str]          # Contradiction Detection
    summary:            str                # Critical Analysis
    insights:           str                # Insight Generator
    final_report:       str                # Report Builder

    # Progress log (printed to console → shows agentic activity)
    log:                Annotated[List[str], operator.add]


# ══════════════════════════════════════════════════════════════════════════════
# AGENT 1 — Query Planner
# Decomposes the research topic into 4-6 targeted sub-queries (multi-hop)
# ══════════════════════════════════════════════════════════════════════════════
def query_planner_agent(state: ResearchState) -> dict:
    print("\n" + "="*60)
    print("🧭 [AGENT 1] QUERY PLANNER — Decomposing research topic...")
    print("="*60)

    topic = state["topic"]
    print(f"   Topic: {topic}")

    messages = [
        SystemMessage(content=(
            "You are a research query strategist. "
            "Given a research topic, decompose it into exactly 5 targeted sub-queries "
            "that together give multi-hop, comprehensive coverage. "
            "Return ONLY a JSON array of 5 strings, no extra text."
        )),
        HumanMessage(content=f"Research topic: {topic}")
    ]

    response = llm.invoke(messages)
    raw = response.content.strip()

    # Parse JSON safely
    try:
        if "```" in raw:
            raw = raw.split("```")[1].replace("json", "").strip()
        sub_queries = json.loads(raw)
    except Exception:
        # Fallback: split by newline
        sub_queries = [line.strip().lstrip("0123456789.-) ") for line in raw.split("\n") if line.strip()][:5]

    print(f"   ✅ Generated {len(sub_queries)} sub-queries:")
    for i, q in enumerate(sub_queries, 1):
        print(f"      {i}. {q}")

    return {
        "sub_queries": sub_queries,
        "log": [f"[Query Planner] Generated {len(sub_queries)} sub-queries for: {topic}"]
    }


# ══════════════════════════════════════════════════════════════════════════════
# AGENT 2 — Contextual Retriever (Web + PDF)
# Fetches web sources via Tavily and extracts PDF content via ChromaDB
# ══════════════════════════════════════════════════════════════════════════════
def retriever_agent(state: ResearchState) -> dict:
    print("\n" + "="*60)
    print("📡 [AGENT 2] CONTEXTUAL RETRIEVER — Fetching sources...")
    print("="*60)

    sub_queries  = state["sub_queries"]
    pdf_paths    = state.get("pdf_paths", [])
    all_sources  = []
    pdf_excerpts = []

    # ── Web Search via Tavily ────────────────────────────────────────────────
    print("   🌐 Web search via Tavily...")
    for i, query in enumerate(sub_queries, 1):
        print(f"      Searching ({i}/{len(sub_queries)}): {query[:70]}...")
        try:
            results = tavily_tool.invoke(query)
            for r in results:
                all_sources.append({
                    "query":   query,
                    "title":   r.get("title", "Unknown"),
                    "url":     r.get("url", ""),
                    "content": r.get("content", "")[:1500],  # cap at 1500 chars per source
                    "score":   r.get("score", 0),
                })
            time.sleep(0.5)   # rate-limit courtesy
        except Exception as e:
            print(f"      ⚠️  Tavily error for query {i}: {e}")

    print(f"   ✅ Retrieved {len(all_sources)} web sources.")

    # ── PDF Semantic Search via ChromaDB ────────────────────────────────────
    if pdf_paths:
        print(f"   📄 Processing {len(pdf_paths)} PDF(s) with ChromaDB...")
        splitter = RecursiveCharacterTextSplitter(chunk_size=800, chunk_overlap=100)
        docs = []
        for path in pdf_paths:
            try:
                loader = PyPDFLoader(path)
                pages  = loader.load()
                chunks = splitter.split_documents(pages)
                docs.extend(chunks)
                print(f"      Loaded: {path} ({len(chunks)} chunks)")
            except Exception as e:
                print(f"      ⚠️  PDF load error: {e}")

        if docs:
            vectorstore = Chroma.from_documents(docs, embeddings)
            for query in sub_queries[:3]:   # query top-3 sub-queries against PDFs
                results = vectorstore.similarity_search(query, k=3)
                for r in results:
                    pdf_excerpts.append(f"[PDF] {r.page_content[:600]}")
            print(f"   ✅ Extracted {len(pdf_excerpts)} PDF excerpts.")
    else:
        print("   ℹ️  No PDFs uploaded — skipping PDF retrieval.")

    return {
        "raw_sources":  all_sources,
        "pdf_excerpts": pdf_excerpts,
        "log": [f"[Retriever] {len(all_sources)} web sources + {len(pdf_excerpts)} PDF excerpts retrieved."]
    }


# ══════════════════════════════════════════════════════════════════════════════
# AGENT 3 — Source Validator
# Scores each source for relevance & credibility; filters out weak ones
# ══════════════════════════════════════════════════════════════════════════════
def source_validator_agent(state: ResearchState) -> dict:
    print("\n" + "="*60)
    print("🔍 [AGENT 3] SOURCE VALIDATOR — Scoring & filtering sources...")
    print("="*60)

    sources = state["raw_sources"]
    topic   = state["topic"]

    if not sources:
        print("   ⚠️  No sources to validate.")
        return {"validated_sources": [], "log": ["[Validator] No sources to validate."]}

    # Send all source titles+snippets in one LLM call for efficiency
    source_list = "\n".join(
        [f"{i+1}. [{s['title']}] {s['content'][:300]}" for i, s in enumerate(sources[:20])]
    )

    messages = [
        SystemMessage(content=(
            "You are a research librarian. For each source, output a JSON array where each element has: "
            '"index" (1-based), "relevance" (0-10), "credibility" (0-10), "keep" (true/false). '
            "Keep a source if relevance >= 5 AND credibility >= 4. Return ONLY the JSON array."
        )),
        HumanMessage(content=f"Research topic: {topic}\n\nSources:\n{source_list}")
    ]

    response  = llm.invoke(messages)
    raw       = response.content.strip()

    try:
        if "```" in raw:
            raw = raw.split("```")[1].replace("json", "").strip()
        scores = json.loads(raw)
    except Exception:
        scores = [{"index": i+1, "keep": True} for i in range(len(sources))]

    keep_indices    = {s["index"]-1 for s in scores if s.get("keep", True)}
    validated       = [s for i, s in enumerate(sources) if i in keep_indices]

    print(f"   ✅ Kept {len(validated)}/{len(sources)} sources after validation.")
    for s in validated[:5]:
        print(f"      • {s['title'][:70]}")

    return {
        "validated_sources": validated,
        "log": [f"[Validator] Kept {len(validated)}/{len(sources)} sources."]
    }


# ══════════════════════════════════════════════════════════════════════════════
# AGENT 4 — Critical Analysis (Summariser + Contradiction Detector)
# Reflection design pattern: summarise → then look for contradictions
# ══════════════════════════════════════════════════════════════════════════════
def critical_analysis_agent(state: ResearchState) -> dict:
    print("\n" + "="*60)
    print("🧐 [AGENT 4] CRITICAL ANALYSIS — Summarising & finding contradictions...")
    print("="*60)

    sources      = state["validated_sources"]
    pdf_excerpts = state.get("pdf_excerpts", [])
    topic        = state["topic"]

    # Build a condensed context (cap at ~6000 chars to stay within context window)
    context_parts = []
    for s in sources:
        context_parts.append(f"Source: {s['title']}\n{s['content'][:400]}")
    context_parts.extend(pdf_excerpts[:5])
    context = "\n\n".join(context_parts)[:6000]

    # ── Summarisation (Reflection) ───────────────────────────────────────────
    print("   📝 Generating summary...")
    sum_messages = [
        SystemMessage(content=(
            "You are a critical research analyst. "
            "Summarise the key findings from the provided sources in 4-6 concise paragraphs. "
            "Highlight the most important facts, data points, and expert opinions. "
            "Be objective and accurate."
        )),
        HumanMessage(content=f"Topic: {topic}\n\nSources:\n{context}")
    ]
    summary = llm.invoke(sum_messages).content
    print("   ✅ Summary generated.")

    # ── Contradiction Detection ──────────────────────────────────────────────
    print("   ⚡ Detecting contradictions...")
    con_messages = [
        SystemMessage(content=(
            "You are a fact-checker. Review the research sources and identify "
            "any contradictions, conflicting claims, or disputed facts. "
            "List each contradiction as a bullet point. If none found, say 'No major contradictions found.'"
        )),
        HumanMessage(content=f"Topic: {topic}\n\nSources:\n{context}")
    ]
    contradictions_text = llm.invoke(con_messages).content
    contradictions      = [line.strip() for line in contradictions_text.split("\n") if line.strip()]

    print(f"   ✅ Found {len(contradictions)} contradiction note(s).")

    return {
        "summary":       summary,
        "contradictions": contradictions,
        "log": [f"[Critical Analysis] Summary done. {len(contradictions)} contradictions noted."]
    }


# ══════════════════════════════════════════════════════════════════════════════
# AGENT 5 — Insight Generator
# Synthesises hypotheses, trends, and key takeaways using reasoning chains
# ══════════════════════════════════════════════════════════════════════════════
def insight_generator_agent(state: ResearchState) -> dict:
    print("\n" + "="*60)
    print("💡 [AGENT 5] INSIGHT GENERATOR — Synthesising trends & hypotheses...")
    print("="*60)

    summary        = state["summary"]
    contradictions = state["contradictions"]
    topic          = state["topic"]

    messages = [
        SystemMessage(content=(
            "You are a strategic research analyst. Based on the research summary and contradictions, "
            "generate: (1) 3 key insights/trends, (2) 2 testable hypotheses, "
            "(3) 3 actionable recommendations, and (4) future research directions. "
            "Use clear reasoning chains. Format with Markdown headers."
        )),
        HumanMessage(content=(
            f"Topic: {topic}\n\n"
            f"Summary:\n{summary}\n\n"
            f"Contradictions:\n{(' ').join(contradictions[:10])}"
        ))
    ]

    insights = llm.invoke(messages).content
    print("   ✅ Insights & hypotheses generated.")

    return {
        "insights": insights,
        "log": ["[Insight Generator] Trends, hypotheses, and recommendations synthesised."]
    }


# ══════════════════════════════════════════════════════════════════════════════
# AGENT 6 — Report Builder
# Compiles all agent outputs into a structured Markdown research report
# ══════════════════════════════════════════════════════════════════════════════
def report_builder_agent(state: ResearchState) -> dict:
    print("\n" + "="*60)
    print("📄 [AGENT 6] REPORT BUILDER — Compiling final report...")
    print("="*60)

    topic          = state["topic"]
    sub_queries    = state["sub_queries"]
    validated      = state["validated_sources"]
    summary        = state["summary"]
    contradictions = state["contradictions"]
    insights       = state["insights"]
    log            = state.get("log", [])
    timestamp      = datetime.now().strftime("%Y-%m-%d %H:%M")

    # Build reference list
    references = ""
    for i, s in enumerate(validated[:10], 1):
        references += f"{i}. [{s['title']}]({s['url']})\n"

    messages = [
        SystemMessage(content=(
            "You are a professional research report writer. "
            "Compile the provided materials into a polished, structured Markdown report. "
            "Use the exact section headers provided. Be thorough but concise."
        )),
        HumanMessage(content=(
            f"Create a report with these sections:\n"
            f"1. Executive Summary\n"
            f"2. Research Scope & Methodology\n"
            f"3. Key Findings\n"
            f"4. Contradictions & Debates\n"
            f"5. Strategic Insights & Recommendations\n"
            f"6. Conclusion\n\n"
            f"Topic: {topic}\n\n"
            f"Sub-queries used: {json.dumps(sub_queries, indent=2)}\n\n"
            f"Summary:\n{summary}\n\n"
            f"Contradictions:\n{(' ').join(contradictions[:10])}\n\n"
            f"Insights:\n{insights}"
        ))
    ]

    report_body = llm.invoke(messages).content

    final_report = f"""# 🔬 Deep Research Report\n**Topic:** {topic}\n**Generated:** {timestamp}\n**Sources analysed:** {len(validated)}\n\n---\n\n{report_body}\n\n---\n\n## 📚 References\n{references}\n\n---\n\n## 🤖 Agent Activity Log\n{(' ').join(['- ' + l for l in log])}\n"""

    print("   ✅ Final report compiled successfully!")
    print("\n" + "="*60)
    print("🎉 PIPELINE COMPLETE")
    print("="*60)

    return {
        "final_report": final_report,
        "log": [f"[Report Builder] Final report compiled. {len(validated)} sources cited."]
    }

# ── Build LangGraph (moved from UBZTW7GBeXOi for self-containment) ───────
from langgraph.graph import StateGraph, END

workflow = StateGraph(ResearchState)

# Register nodes (agents)
workflow.add_node("query_planner",    query_planner_agent)
workflow.add_node("retriever",        retriever_agent)
workflow.add_node("source_validator", source_validator_agent)
workflow.add_node("critical_analysis",critical_analysis_agent)
workflow.add_node("insight_generator",insight_generator_agent)
workflow.add_node("report_builder",   report_builder_agent)

# Define the sequential pipeline
# Query → Retriever → Validator → Analysis → Insight → Report
workflow.set_entry_point("query_planner")
workflow.add_edge("query_planner",    "retriever")
workflow.add_edge("retriever",        "source_validator")
workflow.add_edge("source_validator", "critical_analysis")
workflow.add_edge("critical_analysis","insight_generator")
workflow.add_edge("insight_generator","report_builder")
workflow.add_edge("report_builder",   END)

# Compile
app = workflow.compile()


# ── Direct pipeline run (no UI) ───────────────────────
TEST_TOPIC = "Impact of generative AI on the future of education"

initial_state: ResearchState = {
    "topic":             TEST_TOPIC,
    "pdf_paths":         [],         # add local PDF paths here if needed
    "sub_queries":       [],
    "raw_sources":       [],
    "pdf_excerpts":      [],
    "validated_sources": [],
    "contradictions":    [],
    "summary":           "",
    "insights":          "",
    "final_report":      "",
    "log":               [],
}

result = app.invoke(initial_state)

print("\n\n" + "#"*70)
print("FINAL REPORT")
print("#"*70)
print(result["final_report"])